# 00 · 실습 환경 확인

실습을 시작하기 전에 현재 Jupyter 커널에서 저장소 파일, Python 패키지, CUDA 상태를 확인합니다. 이 노트북은 패키지를 설치하거나 파일을 내려받지 않으며, 학습이나 시스템 설정 변경을 수행하지 않습니다.

**실행 방법:** 위에서 아래로 각 셀을 `Shift + Enter`로 실행합니다. 마지막 요약에서 `확인 필요`가 나온 항목을 강사에게 보여주세요.

- **개인 노트북의 사전 확인:** 아래 `CHECK_EVENT_GPU`를 `False`로 둡니다. CUDA가 없어도 자료를 읽고 준비 상태를 확인할 수 있습니다.
- **안내받은 행사 GPU 세션:** `CHECK_EVENT_GPU`를 `True`로 바꿉니다. 패키지와 CUDA 항목을 함께 검사합니다.

Dockerfile의 기준 이미지는 `nvcr.io/nvidia/physicsnemo/physicsnemo:25.11`입니다. **25.11은 컨테이너 태그**이며 Python 패키지 버전 번호와 다를 수 있습니다. 현재 실행 중인 컨테이너가 이 이미지인지, 행사 설정과 같은지는 운영진의 확인이 필요합니다.

[실습 안내로 돌아가기](README.md)


In [ ]:
# 안내받은 행사 GPU 세션이면 True로 바꾸세요.
CHECK_EVENT_GPU = False

import importlib
import importlib.metadata
import platform
from pathlib import Path

print("확인 모드:", "행사 GPU 세션" if CHECK_EVENT_GPU else "개인 환경 사전 확인")
print("Python:", platform.python_version())
print("운영체제:", platform.system(), platform.machine())
print("기준 컨테이너: nvcr.io/nvidia/physicsnemo/physicsnemo:25.11")
print("현재 컨테이너 태그: 운영진 확인 필요")


## 1. 실습 파일 확인

현재 작업 위치와 상위 폴더에서 이 저장소를 찾습니다. 파일 내용을 바꾸지 않습니다. 필수 파일이 없으면 실습 폴더를 올바르게 열었는지 확인하세요.


In [ ]:
working_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (working_dir, *working_dir.parents)
        if (candidate / "Dockerfile").is_file()
        and (candidate / "tutorial" / "projectile").is_dir()
        and (candidate / "ai4sci").is_dir()
    ),
    None,
)

required_files = [
    "ai4sci/README.md",
    "ai4sci/00_환경확인.ipynb",
    "ai4sci/01_Wave_PINN.ipynb",
    "ai4sci/wave/wave_baseline.py",
    "ai4sci/wave/wave_reference.py",
    "ai4sci/wave/conf/config_wave.yaml",
    "tutorial/projectile/Getting_Started_Projectile.ipynb",
    "tutorial/projectile/source_code/projectile.py",
    "tutorial/projectile/source_code/projectile_eqn.py",
    "tutorial/projectile/source_code/conf/config.yaml",
]
missing_files = []
if repo_root is None:
    print("확인 필요: 저장소를 찾지 못했습니다. Jupyter에서 실습 폴더를 다시 확인하세요.")
    missing_files = required_files.copy()
else:
    relative_dir = working_dir.relative_to(repo_root)
    print("현재 위치(저장소 기준):", str(relative_dir))
    for name in required_files:
        present = (repo_root / name).is_file()
        print(f"{'확인' if present else '확인 필요'}: {name}")
        if not present:
            missing_files.append(name)
    image_lines = [
        line.strip() for line in (repo_root / "Dockerfile").read_text().splitlines()
        if line.strip().upper().startswith("FROM ")
    ]
    for line in image_lines:
        print("Dockerfile 기준:", line)


## 2. 패키지와 PhysicsNeMo Sym 인터페이스 확인

실제 `import`를 수행해 현재 커널에서 패키지를 불러올 수 있는지 확인합니다. 패키지가 설치돼 있어도 다른 커널을 선택했다면 불러오기에 실패할 수 있습니다. 오류가 나면 설치를 시작하기 전에 강사와 실행 환경을 확인하세요.


In [ ]:
package_specs = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("sympy", "sympy"),
    ("torch", "torch"),
    ("physicsnemo", "nvidia-physicsnemo"),
    ("physicsnemo.sym", "nvidia-physicsnemo.sym"),
    ("hydra", "hydra-core"),
    ("omegaconf", "omegaconf"),
]
loaded_modules = {}
package_issues = []
for module_name, distribution_name in package_specs:
    try:
        module = importlib.import_module(module_name)
        loaded_modules[module_name] = module
        try:
            version = importlib.metadata.version(distribution_name)
        except importlib.metadata.PackageNotFoundError:
            version = getattr(module, "__version__", "버전 표기 확인 필요")
        print(f"확인: {module_name} ({version})")
    except Exception as exc:
        package_issues.append(module_name)
        print(f"확인 필요: {module_name} — {type(exc).__name__}")

interface_specs = [
    ("physicsnemo.sym.hydra", "instantiate_arch"),
    ("physicsnemo.sym.solver", "Solver"),
    ("physicsnemo.sym.domain.constraint", "PointwiseInteriorConstraint"),
    ("physicsnemo.sym.domain.constraint", "PointwiseBoundaryConstraint"),
    ("physicsnemo.sym.domain.validator", "PointwiseValidator"),
    ("physicsnemo.sym.domain.inferencer", "PointwiseInferencer"),
    ("physicsnemo.sym.eq.pde", "PDE"),
]
interface_issues = []
for module_name, attribute_name in interface_specs:
    qualified_name = f"{module_name}.{attribute_name}"
    try:
        module = importlib.import_module(module_name)
        getattr(module, attribute_name)
        print(f"확인: {qualified_name}")
    except Exception as exc:
        interface_issues.append(qualified_name)
        print(f"확인 필요: {qualified_name} — {type(exc).__name__}")


## 3. PyTorch와 CUDA 확인

PyTorch가 CUDA를 인식하는지 읽습니다. 큰 텐서나 모델을 GPU에 올리지 않습니다. 여기서 보이는 GPU 번호는 **현재 커널의 논리 번호**입니다. 실제 서버의 물리 GPU 번호와 다를 수 있습니다.

개인 Mac이나 CPU 환경에서 `CUDA 감지: False`는 예상 가능한 결과입니다. 행사 GPU 모드에서는 CUDA가 감지돼야 하므로 강사에게 확인을 요청합니다.


In [ ]:
cuda_available = False
cuda_issue = None
torch_module = loaded_modules.get("torch")
if torch_module is None:
    cuda_issue = "PyTorch 불러오기 실패"
    print("확인 필요: PyTorch를 불러오지 못해 CUDA 상태를 검사하지 못했습니다.")
else:
    try:
        print("PyTorch:", torch_module.__version__)
        print("PyTorch CUDA 빌드:", torch_module.version.cuda)
        cuda_available = torch_module.cuda.is_available()
        print("CUDA 감지:", cuda_available)
        if cuda_available:
            gpu_count = torch_module.cuda.device_count()
            print("현재 커널에서 보이는 GPU 수:", gpu_count)
            for index in range(gpu_count):
                props = torch_module.cuda.get_device_properties(index)
                memory_gib = props.total_memory / (1024 ** 3)
                print(f"논리 GPU {index}: {props.name}, 총 메모리 {memory_gib:.1f} GiB")
        else:
            print("개인 CPU 환경에서는 문서·사전 확인을 진행할 수 있습니다.")
    except Exception as exc:
        cuda_issue = type(exc).__name__
        print("확인 필요: CUDA 상태 조회 중", cuda_issue)


## 4. 결과와 다음 단계

아래 요약은 파일과 기본 환경의 확인 결과입니다. 실제 학습 시간·수렴·검증 그림의 정확도는 핵심 실습 실행과 행사 리허설에서 확인합니다.


In [ ]:
environment_issues = []
if missing_files:
    environment_issues.append(f"필수 파일 {len(missing_files)}개")
if package_issues:
    environment_issues.append("패키지: " + ", ".join(package_issues))
if interface_issues:
    environment_issues.append(f"PhysicsNeMo Sym 인터페이스 {len(interface_issues)}개")
if CHECK_EVENT_GPU and not cuda_available:
    environment_issues.append("행사 GPU 모드에서 CUDA 미감지")
if cuda_issue:
    environment_issues.append("CUDA 조회: " + cuda_issue)

if environment_issues:
    print("확인 필요")
    for item in environment_issues:
        print("-", item)
    print("위 항목을 강사에게 보여주고 지정 커널·실습 폴더를 확인하세요.")
else:
    print("파일·패키지·인터페이스 기본 검사 완료")
    print("행사 GPU 모드:", CHECK_EVENT_GPU, "| CUDA 감지:", cuda_available)
    print("현재 컨테이너와 행사 설정의 일치 여부는 운영진에게 확인하세요.")

print("학습 실행·수렴·행사 배포 상태: 이 노트북의 검사 범위 밖")


## 이어서 진행하기

1. [실습 안내](README.md)에서 오늘의 목표와 진행 순서를 확인합니다.
2. 강사와 [PhysicsNeMo 소개](../tutorial/introduction/Getting_Started_PhysicsNeMo.ipynb) 및 [PINN 기초](../tutorial/introduction/Introductory_Notebook.ipynb)를 읽습니다.
3. 첫 핵심 실습인 [투사체 노트북](../tutorial/projectile/Getting_Started_Projectile.ipynb)으로 이동합니다.
4. 오후에는 [파동 PINN 실습](01_Wave_PINN.ipynb)을 진행합니다.

접속이나 환경 확인이 끝나지 않았다면 강사 시연을 보며 입력·출력·조건과 결과 해석을 먼저 따라갑니다.
